# 03 — Siam-GCAN (Siamese Graph Convolutional Attention Network)

**Family:** Graph CNN + Siamese &nbsp;•&nbsp; **Input:** DE feature pairs `[14 x 5]`

A twin-branch, weight-shared GCN encoder with multi-head self-attention over the
electrode graph. Training uses **trial pairs**: the contrastive term pulls
same-emotion windows together and pushes different-emotion windows apart in the
embedding space, on top of a per-branch cross-entropy term. Inference uses a
single branch, so the shared evaluation harness works unchanged.

## 1. Setup

In [10]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.models import siam_gcan

set_seed(42)
DEVICE = get_device()
# On Apple Silicon this is "mps". Siam-GCAN is tiny (~47k params) and runs on
# DE features [14x5]; if a stage feels slow you can compare against CPU with
#   DEVICE = __import__("torch").device("cpu")

# ----- experiment configuration -----
MODEL_NAME = "SiamGCAN"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.

# Staged evaluation (see src/benchmark_si.py::STAGE_PLAN):
#   1 = sanity     -> 1 held-out subject, <=15 epochs. "Does it train, no bug?"   (minutes)
#   2 = screening  -> 5 evenly-spaced LOSO folds, full epochs + early stopping.   (< ~1 h)
#   3 = final      -> all 23 folds. The number for the thesis table.
# Work up 1 -> 2 -> 3; only run stage 3 once 1 and 2 look right.
STAGE = 1

print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| STAGE:", STAGE)
print("recipe:", B.REGISTRY[MODEL_NAME])
print("stage plan:", B.STAGE_PLAN[STAGE])


device: mps | model: SiamGCAN | label: valence | STAGE: 1
recipe: SISpec(name='SiamGCAN', family='Graph CNN + Siamese', build=<function make_builder.<locals>.build at 0x150a01080>, feature='de', batch_size=128, epochs=45, lr=0.001, val_frac=0.12, channels=None, runner='siam', extra_config={'pairs_per_sample': 2, 'contrastive_lambda': 1.0})
stage plan: {'name': 'sanity', 'n_folds': 1, 'epoch_cap': 15}


## 2. Architecture

`src/models/siam_gcan.py`: `GCANEncoder` = 2 × (`GraphAttnLayer` = learnable-adjacency
GCN propagation + `nn.MultiheadAttention` over electrodes + residual LayerNorm),
mean-pooled over nodes and projected to a 64-D embedding. A small MLP head maps the
embedding to class logits.

**Pair sampling** (`SiamesePairDataset`) draws, for each window, a partner that is
same-class ~50% of the time; `pair_label = 1` iff the two share an emotion class.
Pairs are resampled every epoch.

In [11]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = siam_gcan.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

SiamGCAN(
  (encoder): GCANEncoder(
    (l1): GraphAttnLayer(
      (lin): Linear(in_features=5, out_features=64, bias=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (l2): GraphAttnLayer(
      (lin): Linear(in_features=64, out_features=64, bias=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (drop): Dropout(p=0.3, inplace=False)
    (proj): Linear(in_features=64, out_features=64, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

trainable parameters: 46,922


## 3. Run the subject-independent (LOSO) benchmark

The benchmark uses a single protocol: **23-fold leave-one-subject-out**
(`data/splits/loso_splits.json`). Fold *k* trains a fresh model on 22 subjects and
tests on the held-out 23rd; normalisation is fit on the training subjects only. We
report the **mean ± std across the 23 folds**.

Siam-GCAN's paired objective (`CE(a) + CE(b) + λ·contrastive(emb_a, emb_b)` with
`SiamesePairDataset`) cannot be expressed by the default training loop, so its
custom `fit_fn` lives in `src/benchmark_si.py::make_siam_fit`. `run_one` picks it up
automatically from this model's `REGISTRY` entry — the notebook just calls `run_one`.


In [12]:
# One shared call: 23-fold LOSO (Siamese-pair training), then writes results/SiamGCAN_results.json
res = B.run_one(MODEL_NAME, label=LABEL,  device=DEVICE)
res


=== SiamGCAN  [Graph CNN + Siamese]  full  feature=de batch=128 epochs=45 lr=0.001 folds=all ===
  [LOSO] fold00 (subject_01): acc=0.563 f1=0.553 (108.6s)
  [LOSO] fold01 (subject_02): acc=0.606 f1=0.474 (105.8s)
  [LOSO] fold02 (subject_03): acc=0.611 f1=0.604 (106.8s)
  [LOSO] fold03 (subject_04): acc=0.793 f1=0.719 (115.1s)
  [LOSO] fold04 (subject_05): acc=0.690 f1=0.617 (115.2s)
  [LOSO] fold05 (subject_06): acc=0.535 f1=0.531 (132.4s)
  [LOSO] fold06 (subject_07): acc=0.654 f1=0.396 (37.2s)
  [LOSO] fold07 (subject_08): acc=0.661 f1=0.398 (75.7s)
  [LOSO] fold08 (subject_09): acc=0.557 f1=0.436 (158.0s)
  [LOSO] fold09 (subject_10): acc=0.595 f1=0.571 (171.5s)
  [LOSO] fold10 (subject_11): acc=0.502 f1=0.479 (160.3s)
  [LOSO] fold11 (subject_12): acc=0.488 f1=0.328 (44.3s)
  [LOSO] fold12 (subject_13): acc=0.519 f1=0.453 (153.3s)
  [LOSO] fold13 (subject_14): acc=0.478 f1=0.474 (152.8s)
  [LOSO] fold14 (subject_15): acc=0.646 f1=0.551 (153.8s)
  [LOSO] fold15 (subject_16): acc=0

{'protocol': 'loso',
 'accuracy': 0.5835759226218424,
 'precision': 0.4843859150298765,
 'recall': 0.533635842017881,
 'f1': 0.48989707670563654,
 'accuracy_std': 0.08928665592799444,
 'f1_std': 0.10411844223103892,
 'precision_std': 0.13271147217153606,
 'recall_std': 0.0720957672548464,
 'confusion_matrix': [[9195, 3331], [5538, 3234]],
 'per_fold_scores': [{'accuracy': 0.562634989200864,
   'precision': 0.5825718205849477,
   'recall': 0.5729591836734693,
   'f1': 0.5534224749847283,
   'confusion_matrix': [[194, 296], [109, 327]],
   'n': 926,
   'fold': 0,
   'test_subject': 'subject_01'},
  {'accuracy': 0.6058315334773218,
   'precision': 0.4839924893296411,
   'recall': 0.48911947661947663,
   'f1': 0.4744420446364291,
   'confusion_matrix': [[512, 118], [247, 49]],
   'n': 926,
   'fold': 1,
   'test_subject': 'subject_02'},
  {'accuracy': 0.6112311015118791,
   'precision': 0.6157939008426533,
   'recall': 0.6085761356052438,
   'f1': 0.6038973384030418,
   'confusion_matrix':

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The Siamese training loop and this model's LOSO recipe (feature view, batch
  size, epochs, lr, `pairs_per_sample`, `contrastive_lambda`) live in
  `src/benchmark_si.py` — a single source of truth shared with the CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test (3 folds / few epochs); `LABEL` switches the
  target between valence / arousal / dominance with no other change.
